#Landing para Bronze
Bronze layer com os dados 'as is ".

A unica transformação feita é a padronização de nomes.

`bronze/landing/Chamados_Hora.CSV` -> `bronze.chamados_hora`

In [0]:
from config import (CATALOG,LANDING_PATH,FILE_NAMES_ORIGINAL )

In [0]:
%sql
USE CATALOG v_credit_medallion;
USE SCHEMA default;
-- utilizando os arquivos presentes no landing
LIST "/Volumes/v_credit_medallion/bronze/landing/";

In [0]:
from pyspark.sql.functions import current_timestamp
paths = {}
for file in FILE_NAMES_ORIGINAL:
    name = (file[:-4]).lower() # normalizando os nomes
    paths[name]=f"{LANDING_PATH}/{file}"
for file_name , path in paths.items():
    print(file_name)
    print(path)


In [0]:
df_dict = {}
# talvez seja melhor não inferir o tipo.
# encoding é o Latin-1 (ISO-8859-1)
for name , path in paths.items():
    df_dict[name] = spark.read.option("encoding", "ISO-8859-1").csv(path,header= True, inferSchema = True) 

for normalized_name , dataset in df_dict.items():
    #normalized_name 
    (dataset 
        .withColumn("ingestion_datetime", current_timestamp()) 
        .write.format("delta").mode("overwrite") 
        .saveAsTable(f"{CATALOG}.bronze.{normalized_name}")   
    )